In [1]:
import pandas as pd
import glob

In [2]:
data = pd.read_csv('/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/horizontal_deviatoric_stress_values_depth_0km.csv')

In [3]:
data

,longitude,latitude,horizontal_deviatoric_stress
0,-17.600000,64.500000,0.008736
1,-17.600085,64.500896,0.008996
2,-17.600170,64.501793,0.009260
3,-17.600256,64.502689,0.009528
4,-17.600341,64.503586,0.009802
...,...,...,...
321745,-15.756196,65.592415,0.020009
321746,-15.756248,65.594209,0.019865
321747,-15.756274,65.595107,0.019792
321748,-15.756300,65.596004,0.019720


In [4]:
import numpy as np
from scipy.interpolate import griddata
from pathlib import Path

def interpolate_csv_to_xyz(
    csv_file,
    output_xyz=None,
    spacing=0.002,  # much finer grid spacing (~220 m)
    method="linear",
    region=None,
    value_column=None,
):
    """
    Read one CSV with columns longitude, latitude, value_column and
    interpolate onto a regular lon/lat grid, writing an XYZ file for GMT.

    Parameters
    ----------
    csv_file : str | Path
        Path to CSV file with columns: longitude, latitude, value_column.
    output_xyz : str | Path | None
        Output XYZ file path. If None, writes next to the CSV with
        "_interpolated.xyz" appended to the stem.
    spacing : float
        Grid spacing in degrees (e.g., 0.002 ~= 220 m).
    method : str
        Interpolation method: "linear", "nearest", or "cubic".
    region : tuple | None
        (min_lon, max_lon, min_lat, max_lat). If None, uses data bounds.
    value_column : str | None
        Column name for the value to interpolate. If None, uses the first
        non-longitude/latitude column.
    """
    data = pd.read_csv(csv_file)

    if value_column is None:
        candidates = [c for c in data.columns if c not in ("longitude", "latitude")]
        if not candidates:
            raise ValueError("No value column found (expected a third column).")
        value_column = candidates[0]

    lon = data["longitude"].to_numpy()
    lat = data["latitude"].to_numpy()
    val = data[value_column].to_numpy()

    if region is None:
        min_lon, max_lon = lon.min(), lon.max()
        min_lat, max_lat = lat.min(), lat.max()
    else:
        min_lon, max_lon, min_lat, max_lat = region

    grid_lon = np.arange(min_lon, max_lon + spacing, spacing)
    grid_lat = np.arange(min_lat, max_lat + spacing, spacing)
    grid_lon_mesh, grid_lat_mesh = np.meshgrid(grid_lon, grid_lat)

    grid_val = griddata(
        (lon, lat),
        val,
        (grid_lon_mesh, grid_lat_mesh),
        method=method,
    )

    # Write XYZ: lon lat value (skip NaNs)
    out = np.column_stack(
        [
            grid_lon_mesh.ravel(),
            grid_lat_mesh.ravel(),
            grid_val.ravel(),
        ]
    )
    out = out[~np.isnan(out[:, 2])]

    csv_path = Path(csv_file)
    if output_xyz is None:
        output_xyz = csv_path.with_name(f"{csv_path.stem}_interpolated.xyz")
    else:
        output_xyz = Path(output_xyz)

    np.savetxt(output_xyz, out, fmt="%.6f %.6f %.10f")
    return output_xyz


def interpolate_csvs_to_xyzs(
    csv_files,
    spacing=0.002,  # use finer grid here also
    method="linear",
    region=None,
    value_column=None,
):
    """
    Interpolate each CSV in a list and write an XYZ per file.

    Returns a list of output XYZ paths.
    """
    outputs = []
    for csv_file in csv_files:
        outputs.append(
            interpolate_csv_to_xyz(
                csv_file,
                output_xyz=None,
                spacing=spacing,
                method=method,
                region=region,
                value_column=value_column,
            )
        )
    return outputs

# Example usage:
files = glob.glob('/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/**deviatoric_stress**.csv')
files.extend(glob.glob('/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/**strain**.csv'))
files.extend(glob.glob('/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/**horizontal_principal_stress**.csv'))
# Much finer interpolation
outputs = interpolate_csvs_to_xyzs(files, spacing=0.002, method="linear")

In [5]:
stress_vector_files = glob.glob('/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/**stress_vectors**.csv')

In [6]:
stress_vector_files

['/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/stress_vectors_depth_2km.csv',
 '/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/stress_vectors_depth_5km.csv',
 '/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/stress_vectors_depth_4km.csv',
 '/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/stress_vectors_depth_1km.csv',
 '/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/stress_vectors_depth_3km.csv',
 '/raid2/jam247/A_Askja_Paper/PAPER_FIGURES/Figure_11/model_results/stress_vectors_depth_0km.csv']

In [7]:
from pathlib import Path

def normalize_azimuth(deg):
    """Normalize degrees to [-180, 180)."""
    return ((deg + 180) % 360) - 180


def add_opposite_azimuth_to_csvs(
    csv_files,
    angle_column="direction",
    output_suffix="_with_opposite",
    overwrite=True,
):
    """Append opposite azimuths (angle + 180) for each row in CSVs."""
    outputs = []
    for csv_file in csv_files:
        df = pd.read_csv(csv_file)
        if angle_column not in df.columns:
            raise ValueError(f"Missing '{angle_column}' in {csv_file}")

        df_opposite = df.copy()
        df_opposite[angle_column] = normalize_azimuth(
            df_opposite[angle_column].to_numpy() + 180
        )

        df_out = pd.concat([df, df_opposite], ignore_index=True)

        csv_path = Path(csv_file)
        if overwrite:
            out_path = csv_path
        else:
            out_path = csv_path.with_name(f"{csv_path.stem}{output_suffix}{csv_path.suffix}")

        df_out.to_csv(out_path, index=False)
        outputs.append(out_path)

    return outputs

# Example usage (overwrites in place by default):
outputs = add_opposite_azimuth_to_csvs(stress_vector_files)
# If you want a new file name, set overwrite=False:
# outputs = add_opposite_azimuth_to_csvs(stress_vector_files, overwrite=False)